In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data\truths.csv"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"File not found: {DATA_PATH}")

# Load original data without modifying the source file
df_raw = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False
)

print(f"Original records: {len(df_raw):,}")
print(f"Original columns: {df_raw.shape[1]}")

Original records: 1,051,149
Original columns: 13


## Removing Exact Duplicates

In [2]:
# Create a working copy
df_clean = df_raw.copy()

# Count exact duplicates
duplicates_before = df_clean.duplicated().sum()

# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("DUPLICATE REMOVAL")
print("---------------------------")
print(f"Original records: {len(df_raw):,}")
print(f"Exact duplicates removed: {duplicates_before:,}")
print(f"Remaining records: {len(df_clean):,}")
print(f"Remaining exact duplicates: {df_clean.duplicated().sum():,}")

# Validate the result
assert df_clean.duplicated().sum() == 0

DUPLICATE REMOVAL
---------------------------
Original records: 1,051,149
Exact duplicates removed: 12,324
Remaining records: 1,038,825
Remaining exact duplicates: 0


## Converting Data Types

In [3]:
# Convert identifiers
df_clean["id"] = pd.to_numeric(
    df_clean["id"], errors="raise"
).astype("Int64")

df_clean["author"] = pd.to_numeric(
    df_clean["author"].replace("", pd.NA),
    errors="raise"
).astype("Int64")

# Convert timestamps
for col in ["timestamp", "time_scraped"]:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        format="ISO8601",
        errors="raise"
    )

# Convert boolean columns
boolean_mapping = {"t": True, "f": False}

for col in ["is_retruth", "is_reply"]:
    df_clean[col] = (
        df_clean[col]
        .map(boolean_mapping)
        .astype("boolean")
    )

# Convert engagement metrics
engagement_cols = [
    "like_count",
    "retruth_count",
    "reply_count"
]

for col in engagement_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="raise"
    ).astype("Int64")

print("DATA TYPES AFTER CLEANING")
print("---------------------------")
display(df_clean.dtypes)

print("\nMissing values after conversion:")
display(df_clean.isna().sum())

DATA TYPES AFTER CLEANING
---------------------------


id                          Int64
timestamp          datetime64[us]
time_scraped       datetime64[us]
is_retruth                boolean
is_reply                  boolean
author                      Int64
like_count                  Int64
retruth_count               Int64
reply_count                 Int64
text                       string
external_id                string
url                        string
truth_retruthed            string
dtype: object


Missing values after conversion:


id                   0
timestamp            0
time_scraped         0
is_retruth           0
is_reply             0
author             145
like_count           0
retruth_count        0
reply_count          0
text                 0
external_id          0
url                  0
truth_retruthed      0
dtype: int64

In [4]:
# Identify missing author IDs
missing_author_mask = (
    df_clean["author"].isna()
)

missing_authors = df_clean.loc[missing_author_mask].copy()

print("MISSING AUTHOR ANALYSIS")
print("---------------------------")
print(f"Total records: {len(df_clean):,}")
print(f"Missing authors: {len(missing_authors):,}")
print(f"Missing percentage: {missing_author_mask.mean() * 100:.4f}%")

# Inspect relevant columns
display(
    missing_authors[
        ["id", "author", "external_id", "url"]
    ].head(15)
)

MISSING AUTHOR ANALYSIS
---------------------------
Total records: 1,038,825
Missing authors: 145
Missing percentage: 0.0140%


,id,author,external_id,url
134639,895511,<NA>,truthtent,108864982160272342
134654,895527,<NA>,JackPosobiec,108822381542261053
134663,895535,<NA>,JohnPaulMacIsaac,108814846897812544
134664,895536,<NA>,Lara45,109073924301696705
134679,895573,<NA>,JackPosobiec,108639442992116282
134683,895587,<NA>,MelaniaTrump,109027164115408705
134686,895595,<NA>,NewsMax,109017355982286163
134692,895553,<NA>,DonaldJTrumpJr,108712395933363004
135661,897718,<NA>,doug4gov,109105436245928682
136412,899018,<NA>,PepeLivesMatter,107922117579592120


In [5]:
# Extract usernames from canonical Truth Social URLs
df_clean["username_from_url"] = (
    df_clean["url"]
    .astype("string")
    .str.extract(
        r"(?i)^https?://(?:www\.)?truthsocial\.com/@([^/?#]+)/posts/",
        expand=False
    )
    .str.lower()
)

# Build mappings using records with known author IDs
known_authors = df_clean[
    df_clean["author"].notna()
    & df_clean["username_from_url"].notna()
].copy()

# Count distinct authors per username
username_author_counts = (
    known_authors.groupby("username_from_url")["author"]
    .nunique()
)

# Keep only unambiguous mappings
valid_usernames = username_author_counts[
    username_author_counts == 1
].index

username_to_author = (
    known_authors[
        known_authors["username_from_url"].isin(valid_usernames)
    ]
    .groupby("username_from_url")["author"]
    .first()
)

print("USERNAME-AUTHOR MAPPING")
print("---------------------------")
print(f"Unique usernames: {len(username_author_counts):,}")
print(f"Unambiguous mappings: {len(username_to_author):,}")
print(f"Ambiguous mappings: {(username_author_counts > 1).sum():,}")

USERNAME-AUTHOR MAPPING
---------------------------
Unique usernames: 18,634
Unambiguous mappings: 18,631
Ambiguous mappings: 3


In [6]:
# Candidate username from external_id
external_username = (
    df_clean["external_id"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Consider external_id only when it looks like a username
external_username = external_username.where(
    external_username.str.fullmatch(
        r"[a-z0-9_]+",
        na=False
    )
    & external_username.str.contains(
        r"[a-z_]",
        na=False
    )
)

# Match both potential sources against known author mappings
author_from_url = (
    df_clean["username_from_url"].map(username_to_author)
)

author_from_external = (
    external_username.map(username_to_author)
)

# Detect conflicting candidates
conflicting_candidates = (
    missing_author_mask
    & author_from_url.notna()
    & author_from_external.notna()
    & author_from_url.ne(author_from_external).fillna(False)
)

# Prefer username extracted from the canonical URL
author_candidate = author_from_url.combine_first(
    author_from_external
)

# Exclude conflicting evidence
recoverable_mask = (
    missing_author_mask
    & author_candidate.notna()
    & ~conflicting_candidates
)

print("AUTHOR RECOVERY CANDIDATES")
print("---------------------------")
print(f"Missing authors: {missing_author_mask.sum():,}")
print(f"Recoverable candidates: {recoverable_mask.sum():,}")
print(f"Conflicting candidates: {conflicting_candidates.sum():,}")
print(f"Unresolved: {(missing_author_mask & ~recoverable_mask).sum():,}")

display(
    df_clean.loc[
        recoverable_mask,
        ["id", "author", "external_id", "url"]
    ].assign(
        inferred_author=author_candidate.loc[recoverable_mask]
    ).head(15)
)

AUTHOR RECOVERY CANDIDATES
---------------------------
Missing authors: 145
Recoverable candidates: 58
Conflicting candidates: 0
Unresolved: 87


,id,author,external_id,url,inferred_author
134683,895587,<NA>,MelaniaTrump,109027164115408705,559
134686,895595,<NA>,NewsMax,109017355982286163,145
134692,895553,<NA>,DonaldJTrumpJr,108712395933363004,300
136412,899018,<NA>,PepeLivesMatter,107922117579592120,1343
145236,914237,<NA>,TravisTritt,107849593147323752,206
145368,915764,<NA>,DoItPepe,109119256185776036,9596
145435,914431,<NA>,VeBo1991,108202334611281775,2477
145447,914450,<NA>,RealCarolLeek,108015702747545270,21479
145454,914456,<NA>,jkk4782,107998916362910992,7983
145480,914466,<NA>,ThePatriotVoice,107981749571692604,928


In [7]:
# Preserve information about imputed author IDs
df_clean["author_imputed"] = False

# Fill missing author IDs using validated candidates
df_clean.loc[recoverable_mask, "author"] = (
    author_candidate.loc[recoverable_mask].astype("Int64")
)

# Flag imputed records
df_clean.loc[recoverable_mask, "author_imputed"] = True

# Final validation
print("AUTHOR IMPUTATION RESULTS")
print("---------------------------")
print(f"Missing authors before: {missing_author_mask.sum():,}")
print(f"Authors recovered: {df_clean['author_imputed'].sum():,}")
print(f"Missing authors after: {df_clean['author'].isna().sum():,}")

assert (
    df_clean["author_imputed"].sum()
    + df_clean["author"].isna().sum()
    == missing_author_mask.sum()
)

AUTHOR IMPUTATION RESULTS
---------------------------
Missing authors before: 145
Authors recovered: 58
Missing authors after: 87


In [8]:
# Count supporting observations for each username-author mapping
mapping_support = (
    known_authors.groupby("username_from_url")
    .size()
)

# Get usernames used for imputations
imputed_usernames = (
    df_clean["username_from_url"]
    .combine_first(external_username)
)

# Inspect supporting evidence for recovered authors
validation_df = pd.DataFrame({
    "truth_id": df_clean.loc[df_clean["author_imputed"], "id"],
    "imputed_author": df_clean.loc[df_clean["author_imputed"], "author"],
    "username": imputed_usernames.loc[df_clean["author_imputed"]]
})

validation_df["supporting_records"] = (
    validation_df["username"]
    .map(mapping_support)
)

print("IMPUTATION VALIDATION")
print("---------------------------")

display(
    validation_df.sort_values(
        "supporting_records",
        ascending=True
    ).head(20)
)

print("\nMappings supported by one observation:",
      (validation_df["supporting_records"] == 1).sum())

IMPUTATION VALIDATION
---------------------------


,truth_id,imputed_author,username,supporting_records
145435,914431,2477,vebo1991,1
145454,914456,7983,jkk4782,1
145480,914466,928,thepatriotvoice,1
158668,938012,2081,repgregsteube,1
159769,939902,122,govrondesantis,1
160999,942084,122,govrondesantis,1
160910,941997,207,hodgetwins,1
160937,942039,122,govrondesantis,1
160988,942075,122,govrondesantis,1
161253,942434,9287,patriotflyer,1



Mappings supported by one observation: 11


In [9]:
# Identify the exact username used for each imputation
actual_imputation_username = (
    df_clean["username_from_url"]
    .where(author_from_url.notna())
    .combine_first(
        external_username.where(author_from_external.notna())
    )
)

# Reconstruct the author from the actual mapping
validated_author = actual_imputation_username.map(
    username_to_author
)

# Verify imputed author IDs
imputation_matches = (
    df_clean.loc[df_clean["author_imputed"], "author"]
    .eq(validated_author.loc[df_clean["author_imputed"]])
)

print("IMPUTATION CONSISTENCY CHECK")
print("---------------------------")
print(f"Imputed authors: {df_clean['author_imputed'].sum()}")
print(f"Consistent imputations: {imputation_matches.sum()}")
print(f"Inconsistent imputations: {(~imputation_matches).sum()}")

IMPUTATION CONSISTENCY CHECK
---------------------------
Imputed authors: 58
Consistent imputations: 58
Inconsistent imputations: 0


In [10]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# Define the path to users.xml
USERS_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data\users.xml"
)

if not USERS_PATH.is_file():
    raise FileNotFoundError(f"File not found: {USERS_PATH}")

# Parse the XML file
tree = ET.parse(USERS_PATH)
root = tree.getroot()

print("USERS.XML STRUCTURE")
print("---------------------------")
print("Root tag:", root.tag)
print("Number of direct child elements:", len(root))

# Inspect the first user records
for user in list(root)[:3]:
    print("\nUser tag:", user.tag)
    print("Attributes:", user.attrib)

    for child in user:
        print(f"  {child.tag}: {child.text}")

USERS.XML STRUCTURE
---------------------------
Root tag: data
Number of direct child elements: 487292

User tag: row
Attributes: {}
  id_user: 1238
  timestamp_user_creation: 2022-02-01 00:00:00
  username: jeffgolfs
  follower_count: 886
  following_count: 204
  profile_url: https://truthsocial.com/@jeffgolfs
  finished_follower_scrape: f
  finished_following_scrape: f
  finished_truth_scrape: f

User tag: row
Attributes: {}
  id_user: 50
  timestamp_user_creation: 2022-02-01 00:00:00
  username: MizW
  follower_count: 2365
  following_count: 2929
  profile_url: https://truthsocial.com/@MizW
  finished_follower_scrape: f
  finished_following_scrape: f
  finished_truth_scrape: f

User tag: row
Attributes: {}
  id_user: 6951
  timestamp_user_creation: 2022-02-01 00:00:00
  username: SusanCreech
  follower_count: 870
  following_count: 1232
  profile_url: https://truthsocial.com/@SusanCreech
  finished_follower_scrape: f
  finished_following_scrape: f
  finished_truth_scrape: f


In [11]:
# Extract relevant fields from users.xml
users_records = []

for user in root.findall("row"):
    users_records.append({
        "id_user": user.findtext("id_user"),
        "username": user.findtext("username")
    })

users_df = pd.DataFrame(users_records)

# Normalize usernames
users_df["username_clean"] = (
    users_df["username"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert user IDs to numeric
users_df["id_user"] = pd.to_numeric(
    users_df["id_user"],
    errors="coerce"
).astype("Int64")

# Remove incomplete records
users_valid = users_df.dropna(
    subset=["username_clean", "id_user"]
).copy()

users_valid = users_valid[
    users_valid["username_clean"].ne("")
]

# Identify unambiguous username-to-ID mappings
username_counts = (
    users_valid.groupby("username_clean")["id_user"]
    .nunique()
)

valid_usernames = username_counts[
    username_counts == 1
].index

users_mapping = (
    users_valid[
        users_valid["username_clean"].isin(valid_usernames)
    ]
    .groupby("username_clean")["id_user"]
    .first()
)

print("USERS.XML MAPPING")
print("---------------------------")
print(f"Total users: {len(users_df):,}")
print(f"Valid username mappings: {len(users_mapping):,}")
print(f"Ambiguous usernames: {(username_counts > 1).sum():,}")

USERS.XML MAPPING
---------------------------
Total users: 487,292
Valid username mappings: 487,068
Ambiguous usernames: 80


In [12]:
# Extract username candidates from existing columns
username_from_external = (
    df_clean["external_id"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Select username from URL when available,
# otherwise use external_id
username_candidate = (
    df_clean["username_from_url"]
    .combine_first(username_from_external)
)

# Retrieve author IDs from users.xml
author_from_users = username_candidate.map(users_mapping)

# Check previously imputed authors
previously_imputed = df_clean["author_imputed"]

validation = pd.DataFrame({
    "author_imputed": df_clean.loc[previously_imputed, "author"],
    "author_users_xml": author_from_users.loc[previously_imputed]
})

confirmed = (
    validation["author_users_xml"].notna()
    & validation["author_imputed"].eq(
        validation["author_users_xml"]
    ).fillna(False)
)

conflicting = (
    validation["author_users_xml"].notna()
    & validation["author_imputed"].ne(
        validation["author_users_xml"]
    ).fillna(False)
)

print("PREVIOUS IMPUTATION VALIDATION")
print("---------------------------")
print(f"Previously imputed: {len(validation):,}")
print(f"Confirmed by users.xml: {confirmed.sum():,}")
print(f"Conflicting mappings: {conflicting.sum():,}")
print(f"Not found in users.xml: {validation['author_users_xml'].isna().sum():,}")

display(validation.head(15))

PREVIOUS IMPUTATION VALIDATION
---------------------------
Previously imputed: 58
Confirmed by users.xml: 58
Conflicting mappings: 0
Not found in users.xml: 0


,author_imputed,author_users_xml
134683,559,559
134686,145,145
134692,300,300
136412,1343,1343
145236,206,206
145368,9596,9596
145435,2477,2477
145447,21479,21479
145454,7983,7983
145480,928,928


In [13]:
# Identify remaining missing authors
still_missing = df_clean["author"].isna()

# Find recoverable authors using users.xml
recoverable_from_users = (
    still_missing
    & author_from_users.notna()
)

print("CROSS-DATASET AUTHOR RECOVERY")
print("---------------------------")
print(f"Missing authors before: {still_missing.sum():,}")
print(f"Recoverable from users.xml: {recoverable_from_users.sum():,}")
print(f"Still unresolved: {(still_missing & ~recoverable_from_users).sum():,}")

# Inspect candidate matches before applying changes
display(
    df_clean.loc[
        recoverable_from_users,
        ["id", "external_id", "url", "author"]
    ].assign(
        inferred_author=author_from_users.loc[recoverable_from_users]
    ).head(20)
)

CROSS-DATASET AUTHOR RECOVERY
---------------------------
Missing authors before: 87
Recoverable from users.xml: 87
Still unresolved: 0


,id,external_id,url,author,inferred_author
134639,895511,truthtent,108864982160272342,<NA>,11214
134654,895527,JackPosobiec,108822381542261053,<NA>,143
134663,895535,JohnPaulMacIsaac,108814846897812544,<NA>,2098
134664,895536,Lara45,109073924301696705,<NA>,933
134679,895573,JackPosobiec,108639442992116282,<NA>,143
135661,897718,doug4gov,109105436245928682,<NA>,1413
145137,914125,Elajane,107943822372142377,<NA>,4817
145149,914135,sweetwatergirl,107924695598414362,<NA>,17391
145153,914148,xGoldylocsx,107911657290583986,<NA>,8032
145229,914225,DavidKirkland,107918278278888247,<NA>,40850


In [14]:
# Track which authors were recovered using users.xml
df_clean["author_imputed_users"] = False

# Fill remaining missing author IDs
df_clean.loc[recoverable_from_users, "author"] = (
    author_from_users.loc[recoverable_from_users]
    .astype("Int64")
)

# Mark recovered records
df_clean.loc[recoverable_from_users, "author_imputed_users"] = True

print("FINAL AUTHOR RECOVERY RESULTS")
print("---------------------------")
print(f"Recovered using truths.csv: {df_clean['author_imputed'].sum():,}")
print(f"Recovered using users.xml: {df_clean['author_imputed_users'].sum():,}")
print(f"Total recovered: {(df_clean['author_imputed'] | df_clean['author_imputed_users']).sum():,}")
print(f"Missing authors remaining: {df_clean['author'].isna().sum():,}")

FINAL AUTHOR RECOVERY RESULTS
---------------------------
Recovered using truths.csv: 58
Recovered using users.xml: 87
Total recovered: 145
Missing authors remaining: 0


## Handling Text Problems - MV and <

In [15]:
# Identify missing and malformed text values
text_series = df_clean["text"].astype("string")

# Empty or missing text
missing_text_mask = (
    text_series.isna()
    | text_series.str.strip().eq("")
)

# Text containing only "<"
less_than_mask = (
    text_series.str.strip().eq("<")
)

# Combined problematic records
problematic_text_mask = (
    missing_text_mask | less_than_mask
).fillna(False)

print("TEXT DATA QUALITY ANALYSIS")
print("---------------------------")
print(f"Total Truths: {len(df_clean):,}")
print(f"Missing/empty text: {missing_text_mask.sum():,}")
print(f"Text containing only '<': {less_than_mask.sum():,}")
print(f"Total problematic texts: {problematic_text_mask.sum():,}")
print(f"Percentage affected: {problematic_text_mask.mean() * 100:.2f}%")

# Inspect problematic records
display(
    df_clean.loc[
        problematic_text_mask,
        ["id", "external_id", "url", "author",
         "is_retruth", "is_reply", "text"]
    ].head(20)
)

TEXT DATA QUALITY ANALYSIS
---------------------------
Total Truths: 1,038,825
Missing/empty text: 176
Text containing only '<': 5,423
Total problematic texts: 5,599
Percentage affected: 0.54%


,id,external_id,url,author,is_retruth,is_reply,text
1167,807655,CeceBloomwood,107925568586381393,20054,True,False,
1193,704620,PadingtonRose,108442301408485705,11225,True,False,<
1227,704633,PadingtonRose,108434546171751458,11225,True,False,<
1244,804752,marknicholson,109104919259283807,20030,False,False,
1263,807647,CeceBloomwood,107959581599419915,20054,True,False,
10026,713325,pandyls,108241213535601948,16161,True,False,<
10045,713386,pandyls,108205161088669367,16161,True,False,<
10052,805569,marknicholson,108244712793725159,20030,True,True,
15831,720084,J6_Insurrectionist,108776517385844471,16733,True,False,<
15842,720115,PatriotAR,108642821558716084,8017,True,False,<


In [16]:
# Identify records with usable text
valid_text_mask = (
    ~problematic_text_mask
)

# Count distinct valid texts for each Truth ID
valid_text_counts = (
    df_clean.loc[valid_text_mask]
    .groupby("id")["text"]
    .nunique()
)

# Only retain IDs associated with exactly one valid text
unambiguous_ids = valid_text_counts[
    valid_text_counts == 1
].index

# Build an ID-to-text mapping
text_mapping = (
    df_clean.loc[
        valid_text_mask
        & df_clean["id"].isin(unambiguous_ids)
    ]
    .groupby("id")["text"]
    .first()
)

# Find recovery candidates
recovered_text_candidates = df_clean["id"].map(
    text_mapping
)

recoverable_text_mask = (
    problematic_text_mask
    & recovered_text_candidates.notna()
)

print("INTERNAL TEXT RECOVERY")
print("---------------------------")
print(f"Problematic texts: {problematic_text_mask.sum():,}")
print(f"Recoverable from the same Truth ID: {recoverable_text_mask.sum():,}")
print(f"Still unresolved: {(problematic_text_mask & ~recoverable_text_mask).sum():,}")

display(
    df_clean.loc[
        recoverable_text_mask,
        ["id", "text"]
    ].assign(
        recovered_text=recovered_text_candidates.loc[
            recoverable_text_mask
        ]
    ).head(10)
)

INTERNAL TEXT RECOVERY
---------------------------
Problematic texts: 5,599
Recoverable from the same Truth ID: 0
Still unresolved: 5,599


,id,text,recovered_text


In [17]:
from pathlib import Path

ARFF_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Additional_Data\additional_truths.arff"
)

if not ARFF_PATH.is_file():
    raise FileNotFoundError(f"File not found: {ARFF_PATH}")

print("ADDITIONAL TRUTHS ARFF STRUCTURE")
print("--------------------------------")

with open(ARFF_PATH, "r", encoding="utf-8-sig") as file:
    for line in file:
        line = line.strip()

        if line.lower() == "@data":
            break

        if line and not line.startswith("%"):
            print(line)

ADDITIONAL TRUTHS ARFF STRUCTURE
--------------------------------
@relation trump_truths
@attribute id integer
@attribute timestamp string
@attribute text string
@attribute url string
@attribute reply_count real
@attribute retruth_count real
@attribute like_count real
@attribute external_id string
@attribute author real
@attribute time_scraped string


In [18]:
#%pip install scipy

In [19]:
#%pip install liac-arff

In [22]:
from pathlib import Path
import pandas as pd
import arff
import re

ARFF_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Additional_Data\additional_truths.arff"
)

if not ARFF_PATH.is_file():
    raise FileNotFoundError(f"File not found: {ARFF_PATH}")

# Read ARFF content without modifying the original file
with open(ARFF_PATH, "r", encoding="utf-8-sig") as file:
    arff_content = file.read()

# Escape unsupported \x sequences for the ARFF parser
arff_content_clean = re.sub(
    r"(?<!\\)\\x",
    lambda match: "\\\\x",
    arff_content
)

# Parse ARFF content
arff_data = arff.loads(arff_content_clean)

# Convert to DataFrame
column_names = [
    attribute[0]
    for attribute in arff_data["attributes"]
]

additional_df = pd.DataFrame(
    arff_data["data"],
    columns=column_names
)

print("ADDITIONAL TRUTHS DATASET")
print("---------------------------")
print(f"Total records: {len(additional_df):,}")
print(f"Columns: {additional_df.shape[1]}")

display(additional_df.head())
display(additional_df.dtypes)

ADDITIONAL TRUTHS DATASET
---------------------------
Total records: 35,222
Columns: 10


,id,timestamp,text,url,reply_count,retruth_count,like_count,external_id,author,time_scraped
0,117323511508375200,2026-09-24 01:51:54,,https://truthsocial.com/@realDonaldTrump/11732...,792.0,1849.0,7259.0,117323511508375196,27.0,2026-09-20 14:00:00.000000
1,117323155721195616,2026-09-24 00:21:25,"The story that the White House, or me, want to...",https://truthsocial.com/@realDonaldTrump/11732...,1329.0,3534.0,12560.0,117323155721195611,27.0,2026-09-20 14:00:00.000000
2,117323148147220272,2026-09-24 00:19:29,BIG DAY FOR BOEING AND AMERICAN MANUFACTURING!...,https://truthsocial.com/@realDonaldTrump/11732...,510.0,2714.0,10677.0,117323148147220269,27.0,2026-09-20 14:00:00.000000
3,117322092055919840,2026-09-23 19:50:55,,https://truthsocial.com/@realDonaldTrump/11732...,1344.0,2740.0,11287.0,117322092055919832,27.0,2026-09-20 14:00:00.000000
4,117322065119445872,2026-09-23 19:44:04,"The Great Gary Player, Bryson DeChambeau, Mika...",https://truthsocial.com/@realDonaldTrump/11732...,505.0,2156.0,10284.0,117322065119445871,27.0,2026-09-20 14:00:00.000000


id                 int64
timestamp            str
text                 str
url                  str
reply_count      float64
retruth_count    float64
like_count       float64
external_id          str
author           float64
time_scraped         str
dtype: object

In [23]:
print("Unique IDs:", additional_df["id"].nunique())
print("Missing texts:", additional_df["text"].isna().sum())
print("Total records:", len(additional_df))

Unique IDs: 35221
Missing texts: 0
Total records: 35222


In [24]:
# Normalize ARFF IDs
additional_df["id"] = pd.to_numeric(
    additional_df["id"],
    errors="coerce"
).astype("Int64")

# Check text quality in the ARFF dataset
arff_text = additional_df["text"].astype("string")

valid_arff_text = (
    arff_text.notna()
    & arff_text.str.strip().ne("")
    & arff_text.str.strip().ne("<")
).fillna(False)

print("ARFF TEXT QUALITY")
print("---------------------------")
print(f"Total records: {len(additional_df):,}")
print(f"Usable texts: {valid_arff_text.sum():,}")
print(f"Problematic texts: {(~valid_arff_text).sum():,}")

# Compare IDs between datasets
problematic_ids = set(
    df_clean.loc[problematic_text_mask, "id"].dropna()
)

additional_ids = set(additional_df["id"].dropna())

matching_ids = problematic_ids & additional_ids

print("\nARFF TEXT RECOVERY ANALYSIS")
print("---------------------------")
print(f"Problematic Truths: {len(problematic_ids):,}")
print(f"Matching Truth IDs: {len(matching_ids):,}")
print(f"Unmatched Truth IDs: {len(problematic_ids - additional_ids):,}")

# Inspect matching records
comparison = df_clean.loc[
    problematic_text_mask,
    ["id", "author", "external_id", "url", "text"]
].merge(
    additional_df.loc[
        additional_df["id"].isin(matching_ids) & valid_arff_text,
        ["id", "author", "external_id", "url", "text"]
    ],
    on="id",
    how="inner",
    suffixes=("_original", "_arff")
)

print(f"\nPotential recovery candidates: {len(comparison):,}")

display(comparison.head(10))

ARFF TEXT QUALITY
---------------------------
Total records: 35,222
Usable texts: 27,960
Problematic texts: 7,262

ARFF TEXT RECOVERY ANALYSIS
---------------------------
Problematic Truths: 5,599
Matching Truth IDs: 0
Unmatched Truth IDs: 5,599

Potential recovery candidates: 0


,id,author_original,external_id_original,url_original,text_original,author_arff,external_id_arff,url_arff,text_arff


In [25]:
print("TRUTHS.CSV IDENTIFIERS")
print("---------------------------")

display(
    df_clean.loc[
        problematic_text_mask,
        ["id", "external_id", "url", "text"]
    ].head(10)
)

print("\nARFF IDENTIFIERS")
print("---------------------------")

display(
    additional_df[
        ["id", "external_id", "url", "text"]
    ].head(10)
)

TRUTHS.CSV IDENTIFIERS
---------------------------


,id,external_id,url,text
1167,807655,CeceBloomwood,107925568586381393,
1193,704620,PadingtonRose,108442301408485705,<
1227,704633,PadingtonRose,108434546171751458,<
1244,804752,marknicholson,109104919259283807,
1263,807647,CeceBloomwood,107959581599419915,
10026,713325,pandyls,108241213535601948,<
10045,713386,pandyls,108205161088669367,<
10052,805569,marknicholson,108244712793725159,
15831,720084,J6_Insurrectionist,108776517385844471,<
15842,720115,PatriotAR,108642821558716084,<



ARFF IDENTIFIERS
---------------------------


,id,external_id,url,text
0,117323511508375200,117323511508375196,https://truthsocial.com/@realDonaldTrump/11732...,
1,117323155721195616,117323155721195611,https://truthsocial.com/@realDonaldTrump/11732...,"The story that the White House, or me, want to..."
2,117323148147220272,117323148147220269,https://truthsocial.com/@realDonaldTrump/11732...,BIG DAY FOR BOEING AND AMERICAN MANUFACTURING!...
3,117322092055919840,117322092055919832,https://truthsocial.com/@realDonaldTrump/11732...,
4,117322065119445872,117322065119445871,https://truthsocial.com/@realDonaldTrump/11732...,"The Great Gary Player, Bryson DeChambeau, Mika..."
5,117320581360547088,117320581360547091,https://truthsocial.com/@realDonaldTrump/11732...,Victor Davis Hanson on Canada — A MUST SEE! Pr...
6,117320509296515184,117320509296515188,https://truthsocial.com/@realDonaldTrump/11732...,Canada is destroying itself by allowing millio...
7,117320385350427392,117320385350427395,https://truthsocial.com/@realDonaldTrump/11732...,Thank you to the Great Robert Greenway and Jes...
8,117318371569831040,117318371569831036,https://truthsocial.com/@realDonaldTrump/11731...,Ruddy: 'A Rising Trump Lifts All Boats': https...
9,117318370203435200,117318370203435194,https://truthsocial.com/@realDonaldTrump/11731...,Azerbaijan's Hajiyev to Newsmax: Trump Peacema...


In [26]:
import re

def extract_post_id(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    # Numeric post identifier
    if re.fullmatch(r"\d+", value):
        return value

    # Truth Social post URL
    match = re.search(
        r"truthsocial\.com/@[^/]+/posts/(\d+)",
        value,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1)

    return pd.NA


# Extract post IDs from both datasets
df_clean["post_id"] = (
    df_clean["url"].map(extract_post_id).astype("string")
)

additional_df["post_id"] = (
    additional_df["url"].map(extract_post_id).astype("string")
)

# Compare problematic Truth post IDs with ARFF post IDs
problematic_post_ids = set(
    df_clean.loc[
        problematic_text_mask, "post_id"
    ].dropna()
)

arff_post_ids = set(
    additional_df["post_id"].dropna()
)

matching_post_ids = problematic_post_ids & arff_post_ids

print("POST ID MATCHING ANALYSIS")
print("---------------------------")
print(f"Problematic Truth post IDs: {len(problematic_post_ids):,}")
print(f"ARFF post IDs: {len(arff_post_ids):,}")
print(f"Matching post IDs: {len(matching_post_ids):,}")

POST ID MATCHING ANALYSIS
---------------------------
Problematic Truth post IDs: 4,417
ARFF post IDs: 0
Matching post IDs: 0


In [27]:
print("ARFF IDENTIFIER INSPECTION")
print("---------------------------")

# Display identifiers from the ARFF dataset
display(
    additional_df[
        ["id", "external_id", "url", "author"]
    ].head(20)
)

# Count numeric values in each column
for col in ["external_id", "url"]:

    values = additional_df[col].astype("string").str.strip()

    numeric_count = values.str.fullmatch(
        r"\d+", na=False
    ).sum()

    canonical_url_count = values.str.contains(
        r"truthsocial\.com",
        case=False,
        na=False
    ).sum()

    print(f"\nColumn: {col}")
    print(f"Numeric values: {numeric_count:,}")
    print(f"Truth Social URLs: {canonical_url_count:,}")
    print(f"Unique values: {values.nunique():,}")

ARFF IDENTIFIER INSPECTION
---------------------------


,id,external_id,url,author
0,117323511508375200,117323511508375196,https://truthsocial.com/@realDonaldTrump/11732...,27.0
1,117323155721195616,117323155721195611,https://truthsocial.com/@realDonaldTrump/11732...,27.0
2,117323148147220272,117323148147220269,https://truthsocial.com/@realDonaldTrump/11732...,27.0
3,117322092055919840,117322092055919832,https://truthsocial.com/@realDonaldTrump/11732...,27.0
4,117322065119445872,117322065119445871,https://truthsocial.com/@realDonaldTrump/11732...,27.0
5,117320581360547088,117320581360547091,https://truthsocial.com/@realDonaldTrump/11732...,27.0
6,117320509296515184,117320509296515188,https://truthsocial.com/@realDonaldTrump/11732...,27.0
7,117320385350427392,117320385350427395,https://truthsocial.com/@realDonaldTrump/11732...,27.0
8,117318371569831040,117318371569831036,https://truthsocial.com/@realDonaldTrump/11731...,27.0
9,117318370203435200,117318370203435194,https://truthsocial.com/@realDonaldTrump/11731...,27.0



Column: external_id
Numeric values: 35,222
Truth Social URLs: 0
Unique values: 35,222

Column: url
Numeric values: 0
Truth Social URLs: 35,222
Unique values: 35,222


In [28]:
# Normalize post identifiers as strings
arff_post_ids = (
    additional_df["external_id"]
    .astype("string")
    .str.strip()
)

# Use post IDs previously extracted from truths.csv
truth_post_ids = (
    df_clean["post_id"]
    .astype("string")
    .str.strip()
)

# Compare problematic Truths with ARFF records
problematic_post_ids = set(
    truth_post_ids.loc[problematic_text_mask].dropna()
)

additional_post_ids = set(arff_post_ids.dropna())

matching_post_ids = (
    problematic_post_ids & additional_post_ids
)

print("CROSS-DATASET POST ID MATCHING")
print("--------------------------------")
print(f"Problematic Truth post IDs: {len(problematic_post_ids):,}")
print(f"ARFF post IDs: {len(additional_post_ids):,}")
print(f"Matching post IDs: {len(matching_post_ids):,}")

# Identify candidate ARFF records with usable text
recovery_candidates = additional_df.loc[
    arff_post_ids.isin(matching_post_ids)
    & valid_arff_text
].copy()

print(f"Potential records with usable text: {len(recovery_candidates):,}")

display(
    recovery_candidates[
        ["id", "external_id", "url", "text"]
    ].head(10)
)

CROSS-DATASET POST ID MATCHING
--------------------------------
Problematic Truth post IDs: 4,417
ARFF post IDs: 35,222
Matching post IDs: 13
Potential records with usable text: 10


,id,external_id,url,text
34298,109111035929718448,109111035929718455,https://truthsocial.com/@realDonaldTrump/10911...,RT @loudobbsRobert Jeffress joins us on #TheGr...
34318,109102004910960720,109102004910960727,https://truthsocial.com/@realDonaldTrump/10910...,https://www.aa.com.tr/en/americas/trump-calls-...
34397,109082720069591440,109082720069591446,https://truthsocial.com/@realDonaldTrump/10908...,“A former president can keep whatever presiden...
34399,109082626904067504,109082626904067503,https://truthsocial.com/@realDonaldTrump/10908...,RT @realDonaldTrumpThis is the real story on t...
34414,109074021311022832,109074021311022825,https://truthsocial.com/@realDonaldTrump/10907...,RT: https://truthsocial.com/users/KickDreaming...
34443,109067943225951888,109067943225951882,https://truthsocial.com/@realDonaldTrump/10906...,https://www.breitbart.com/entertainment/2022/0...
34481,109054369938394272,109054369938394267,https://truthsocial.com/@realDonaldTrump/10905...,"“Exclusive — Tom Homan: If Trump Wins, Many ‘P..."
34499,109048404143018336,109048404143018341,https://truthsocial.com/@realDonaldTrump/10904...,“FBI hero paying the price for exposing unjust...
34591,109027164115408704,109027164115408705,https://truthsocial.com/@realDonaldTrump/10902...,RT @MelaniaTrumpRIP Her Majesty Queen Elizabet...
34617,109017355982286160,109017355982286163,https://truthsocial.com/@realDonaldTrump/10901...,RT @NewsMaxJUST IN: Former President Donald Tr...


In [29]:
from pathlib import Path
import pandas as pd

MEDIA_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data\media.json"
)

media_df = pd.read_json(MEDIA_PATH)

# Compare Truth IDs as numeric identifiers
media_truth_ids = set(
    pd.to_numeric(
        media_df["truth_id"], errors="coerce"
    ).dropna()
)

problematic_with_media = (
    df_clean.loc[problematic_text_mask, "id"]
    .isin(media_truth_ids)
)

print("PROBLEMATIC TEXTS AND MEDIA")
print("---------------------------")
print(f"Total problematic texts: {problematic_text_mask.sum():,}")
print(f"Associated with media: {problematic_with_media.sum():,}")
print(f"Without media association: {(~problematic_with_media).sum():,}")

PROBLEMATIC TEXTS AND MEDIA
---------------------------
Total problematic texts: 5,599
Associated with media: 1,544
Without media association: 4,055


In [30]:
import pandas as pd
import html

# Preserve original text
original_text = df_clean["text"].astype("string")

# Normalize HTML entities without overwriting the source
df_clean["text_clean"] = original_text.map(
    lambda x: html.unescape(x) if pd.notna(x) else pd.NA
).astype("string")

# Identify problematic texts
empty_mask = (
    original_text.isna()
    | original_text.str.strip().eq("")
).fillna(False)

malformed_mask = (
    original_text.str.strip().eq("<")
).fillna(False)

# Classify text quality
df_clean["text_quality_status"] = "valid"

df_clean.loc[
    empty_mask, "text_quality_status"
] = "empty"

df_clean.loc[
    malformed_mask, "text_quality_status"
] = "malformed"

# Replace unusable content with missing values
df_clean.loc[
    empty_mask | malformed_mask,
    "text_clean"
] = pd.NA

print("TEXT CLEANING RESULTS")
print("---------------------------")
print("Text quality distribution:")
display(df_clean["text_quality_status"].value_counts())

print("\nMissing values in text_clean:",
      df_clean["text_clean"].isna().sum())

print("\nOriginal text preserved:",
      df_clean["text"].equals(original_text))

TEXT CLEANING RESULTS
---------------------------
Text quality distribution:


text_quality_status
valid        1033226
malformed       5423
empty            176
Name: count, dtype: int64


Missing values in text_clean: 5599

Original text preserved: True


In [31]:
# Identify Truths associated with media
df_clean["has_media"] = (
    df_clean["id"].isin(media_truth_ids)
)

# Combine text quality with media availability
text_media_summary = pd.crosstab(
    df_clean["text_quality_status"],
    df_clean["has_media"]
)

print("TEXT QUALITY VS MEDIA")
print("---------------------------")

display(text_media_summary)

TEXT QUALITY VS MEDIA
---------------------------


has_media,False,True
text_quality_status,,
empty,129,47
malformed,3926,1497
valid,896921,136305


In [32]:
# Verify the number of problematic texts
assert (
    df_clean["text_quality_status"].isin(
        ["empty", "malformed"]
    ).sum() == 5599
)

# Verify that unusable texts are missing in text_clean
assert df_clean.loc[
    df_clean["text_quality_status"].isin(
        ["empty", "malformed"]
    ),
    "text_clean"
].isna().all()

# Verify that valid texts remain available
assert df_clean.loc[
    df_clean["text_quality_status"] == "valid",
    "text_clean"
].notna().all()

print("Text cleaning validation passed.")

Text cleaning validation passed.


In [33]:
# Final text cleaning validation
problematic_status = df_clean["text_quality_status"].isin(
    ["empty", "malformed"]
)

assert (
    df_clean.loc[problematic_status, "text_clean"]
    .isna().all()
)

assert (
    df_clean.loc[~problematic_status, "text_clean"]
    .notna().all()
)

assert (
    problematic_status.sum()
    == df_clean["text_clean"].isna().sum()
)

print("TEXT CLEANING VALIDATION")
print("---------------------------")
print("All checks passed.")
print(f"Usable texts: {df_clean['text_clean'].notna().sum():,}")
print(f"Unusable texts: {df_clean['text_clean'].isna().sum():,}")

TEXT CLEANING VALIDATION
---------------------------
All checks passed.
Usable texts: 1,033,226
Unusable texts: 5,599
